# Preprocessing Pipeline - IGAR Sentiment Classification

**Dataset:** Indonesian Government App Reviews (IGAR)
**Input:** `data/Rating_labeled_raw.csv`
**Output:** `data/Rating_labeled_processed.csv`
**Report:** `docs/preprocessing_report.md`

Pipeline ini menghasilkan dataset yang sudah dibersihkan secara konservatif (emoji distrip, slang dinormalisasi, Unicode didekomposisi), menyediakan dua varian teks fitur (`content_clean` dan `content_clean_stopword`), serta dilengkapi metadata grup duplikasi untuk evaluasi cross-app.

## 1. Import dan Konfigurasi

In [1]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

RAW_PATH = '../data/Rating_labeled_raw.csv'
OUTPUT_CSV = '../data/Rating_labeled_processed.csv'
OUTPUT_REPORT = '../docs/preprocessing_report.md'


## 2. Load Raw Data dan Validasi Skema

In [2]:
# Load raw data
df_raw = pd.read_csv(RAW_PATH)
n_raw = len(df_raw)

print(f'Total baris raw: {n_raw:,}')
print(f'Total kolom: {len(df_raw.columns)}')
print(f'Kolom: {df_raw.columns.tolist()}')

# Validasi kolom wajib
required_cols = ['Unnamed: 0', 'app', 'content', 'score', 'at', 'labelScoreBase']
for c in required_cols:
    assert c in df_raw.columns, f'Kolom wajib tidak ditemukan: {c}'

# Schema & Missing values
schema = pd.DataFrame({
    'Tipe Data': [str(t) for t in df_raw.dtypes],
    'Non-Null': df_raw.notnull().sum().values,
    'Null': df_raw.isnull().sum().values,
    'Null (%)': (df_raw.isnull().sum().values / n_raw * 100).round(4)
}, index=df_raw.columns)
display(schema)

# Validasi label mapping: score 1-2 = Negative, 3 = Neutral, 4-5 = Positive
label_map = {1: 'Negative', 2: 'Negative', 3: 'Neutral', 4: 'Positive', 5: 'Positive'}
derived_label = df_raw['score'].map(label_map)
n_mismatch = (derived_label != df_raw['labelScoreBase']).sum()
print(f'\nMismatch mapping rating vs labelScoreBase: {n_mismatch}')
assert n_mismatch == 0, 'Label mapping tidak konsisten!'

# Distribusi label dan aplikasi raw
print('\nDistribusi label raw:')
display(df_raw['labelScoreBase'].value_counts().to_frame('Jumlah').assign(**{
    'Persentase (%)': lambda x: (x['Jumlah'] / n_raw * 100).round(2)
}))

print('\nDistribusi aplikasi raw:')
display(df_raw['app'].value_counts().to_frame('Jumlah').assign(**{
    'Persentase (%)': lambda x: (x['Jumlah'] / n_raw * 100).round(2)
}))


Total baris raw: 617,722
Total kolom: 8
Kolom: ['Unnamed: 0', 'app', 'content', 'translation', 'score', 'at', 'appVersion', 'labelScoreBase']


,Tipe Data,Non-Null,Null,Null (%)
Unnamed: 0,int64,617722,0,0.0000
app,str,617722,0,0.0000
content,str,617696,26,0.0042
translation,str,609947,7775,1.2587
score,int64,617722,0,0.0000
at,str,617722,0,0.0000
appVersion,str,527319,90403,14.6349
labelScoreBase,str,617722,0,0.0000



Mismatch mapping rating vs labelScoreBase: 0

Distribusi label raw:


,Jumlah,Persentase (%)
labelScoreBase,,
Positive,336449,54.47
Negative,246898,39.97
Neutral,34375,5.56



Distribusi aplikasi raw:


,Jumlah,Persentase (%)
app,,
JMO,200000,32.38
satusehat,190696,30.87
mobileJKN,122103,19.77
pertamina,96291,15.59
KAI,7311,1.18
BMKG,1321,0.21


## 3. Dataframe Kerja dan Review ID

In [3]:
# Buat dataframe kerja dengan review_id stabil dari indeks raw
df = pd.DataFrame()
df['review_id'] = df_raw['Unnamed: 0']
df['app'] = df_raw['app']
df['content_original'] = df_raw['content']
df['score'] = df_raw['score']
df['label'] = df_raw['score'].map(label_map)
df['at'] = df_raw['at']

print(f'review_id unik: {df["review_id"].is_unique}')
print(f'Rentang review_id: {df["review_id"].min()} - {df["review_id"].max()}')
print(f'Baris content NULL: {df["content_original"].isnull().sum()}')


review_id unik: True
Rentang review_id: 0 - 617721
Baris content NULL: 26


## 4. Kamus Slang dan Daftar Stopwords

- **Emoji:** Dihapus langsung tanpa pemetaan untuk menghindari distorsi semantik/sarkasme.
- **Slang:** Normalisasi singkatan dan kata tidak baku dengan kamus terdokumentasi (kata `'dr'` dipertahankan untuk mencegah konflik dengan 'dokter' pada domain kesehatan).
- **Stopwords:** Daftar stopwords bahasa Indonesia untuk kolom sekunder `content_clean_stopword`, dengan **mengecualikan kata negasi** (`tidak`, `bukan`, `jangan`, `belum`, `kurang`) dan kata penguat sentimen.

In [4]:
# Kamus normalisasi slang bahasa Indonesia
SLANG_DICT = {
    # Negasi (krusial untuk polaritas sentimen)
    'tdk': 'tidak', 'gak': 'tidak', 'ga': 'tidak', 'gk': 'tidak',
    'nggak': 'tidak', 'ngga': 'tidak', 'engga': 'tidak', 'enggak': 'tidak',
    'gx': 'tidak', 'tida': 'tidak', 'tdak': 'tidak',
    'jgn': 'jangan', 'jng': 'jangan',
    'blm': 'belum', 'blum': 'belum',
    
    # Kata hubung & preposisi ('dr' tidak dimasukkan untuk cegah konflik 'dokter')
    'yg': 'yang', 'dgn': 'dengan', 'dg': 'dengan', 'dng': 'dengan',
    'utk': 'untuk', 'krn': 'karena', 'krna': 'karena', 'karna': 'karena',
    'tp': 'tapi', 'tpi': 'tapi',
    'dri': 'dari',
    'sm': 'sama', 'dlm': 'dalam',
    'spt': 'seperti', 'sprti': 'seperti',
    'stlh': 'setelah', 'sblm': 'sebelum',
    'klo': 'kalau', 'kalo': 'kalau', 'klu': 'kalau',
    
    # Kata kerja umum
    'bs': 'bisa', 'bsa': 'bisa',
    'sdh': 'sudah', 'udh': 'sudah', 'udah': 'sudah',
    'hrs': 'harus', 'hrus': 'harus',
    'dpt': 'dapat',
    'pke': 'pakai', 'pake': 'pakai',
    'smpe': 'sampai', 'sampe': 'sampai',
    'trs': 'terus', 'trus': 'terus',
    'cb': 'coba',
    
    # Kata sifat & intensitas sentimen
    'bgus': 'bagus', 'bgs': 'bagus',
    'jlek': 'jelek', 'jlk': 'jelek',
    'bner': 'benar', 'bnr': 'benar',
    'lmyn': 'lumayan', 'lumyan': 'lumayan',
    'mantep': 'mantap', 'mantab': 'mantap',
    'bgt': 'banget', 'bngt': 'banget',
    'krg': 'kurang',
    'bnyk': 'banyak', 'byk': 'banyak',
    
    # Pronomina (diselaraskan ke 'saya' dan 'kamu')
    'sy': 'saya', 'ak': 'saya', 'aku': 'saya', 'gue': 'saya', 'gw': 'saya',
    'lu': 'kamu', 'km': 'kamu',
    'org': 'orang', 'orng': 'orang',
    
    # Adverbia & kata umum
    'jg': 'juga', 'lg': 'lagi',
    'msh': 'masih',
    'skrg': 'sekarang', 'skr': 'sekarang',
    'srng': 'sering', 'sllu': 'selalu',
    'emg': 'memang', 'emang': 'memang',
    'aja': 'saja', 'doang': 'saja',
    'cmn': 'cuma',
    'gmn': 'bagaimana', 'gmna': 'bagaimana', 'gimana': 'bagaimana',
    'knp': 'kenapa',
    'mgkn': 'mungkin', 'mngkn': 'mungkin',
    'pdhl': 'padahal', 'mslh': 'masalah',
    'mksh': 'terima kasih', 'makasih': 'terima kasih',
    'thx': 'terima kasih', 'tq': 'terima kasih', 'trims': 'terima kasih',
    'aplk': 'aplikasi', 'apk': 'aplikasi',
    'blh': 'boleh',
    
    # Reduksi varian elongasi umum
    'baguus': 'bagus', 'jelekk': 'jelek', 'mantapp': 'mantap',
    'kerenn': 'keren', 'okee': 'oke', 'parahh': 'parah',
    'lemott': 'lemot', 'susahh': 'susah', 'lambatt': 'lambat',
    'payahh': 'payah'
}

# Stopwords bahasa Indonesia (kata negasi sengaja DIKECUALIKAN agar polaritas tidak rusak)
INDONESIAN_STOPWORDS = {
    'yang', 'untuk', 'pada', 'ke', 'para', 'namun', 'menurut', 'antara', 'dia', 'dua',
    'ia', 'seperti', 'jika', 'sehingga', 'kembali', 'dan', 'ini', 'karena', 'oleh',
    'saat', 'olehnya', 'adalah', 'bahwa', 'akan', 'atau', 'ada', 'mereka',
    'sudah', 'saya', 'kita', 'kami', 'kamu', 'anda', 'tersebut', 'itu',
    'dari', 'dalam', 'dengan', 'saja', 'di', 'terhadap', 'secara', 'harus', 'kalau',
    'bahkan', 'sampai', 'terus', 'lagi', 'juga', 'setelah', 'sebelum', 'hanya', 'cuma',
    'bagaimana', 'kenapa', 'mengapa', 'apa', 'siapa', 'dimana', 'kapan', 'kok', 'padahal',
    'mungkin', 'tentu', 'tentang', 'lah', 'kah', 'pun', 'deh', 'dong', 'sih', 'nih', 'tuh',
    'kan', 'yah', 'ya', 'terima', 'kasih'
}

print(f'Total entri kamus slang: {len(SLANG_DICT)}')
print(f'Total stopwords terdaftar: {len(INDONESIAN_STOPWORDS)}')


Total entri kamus slang: 112
Total stopwords terdaftar: 78


## 5. Eksekusi Cleaning Teks dan Penghapusan Stopwords

In [5]:
def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    # 1. Dekomposisi Unicode NFKD (mempertahankan font stylized ke huruf Latin standar)
    t = unicodedata.normalize('NFKD', text)
    # 2. Case folding
    t = t.lower()
    # 3. Hapus URL
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    # 4. Hapus Email
    t = re.sub(r'\S+@\S+\.\S+', ' ', t)
    # 5. Hapus karakter non-alfanumerik (emoji & tanda baca otomatis terhapus)
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    # 6. De-elongasi karakter (3+ -> 2)
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)
    # 7. Normalisasi slang
    words = t.split()
    words = [SLANG_DICT.get(w, w) for w in words]
    t = ' '.join(words)
    # 8. Trim whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def remove_stopwords(text):
    if not text:
        return ''
    words = text.split()
    filtered = [w for w in words if w not in INDONESIAN_STOPWORDS]
    return ' '.join(filtered)

# Terapkan cleaning
df['content_clean'] = df['content_original'].apply(clean_text)

# Statistik panjang sebelum filtering
clean_words = df['content_clean'].str.split().apply(len)
print(f'Rata-rata kata per ulasan: {clean_words.mean():.2f}')
print(f'Median kata per ulasan: {clean_words.median():.0f}')


Rata-rata kata per ulasan: 8.49
Median kata per ulasan: 4


## 6. Filtering Baris Kosong / Non-Usable

In [6]:
# Identifikasi baris kosong
mask_null = df['content_original'].isnull()
mask_empty = df['content_clean'].str.strip() == ''
mask_filter = mask_null | mask_empty

n_null = int(mask_null.sum())
n_empty = int(mask_empty.sum())
n_filtered = int(mask_filter.sum())

print(f'Baris content_original NULL: {n_null}')
print(f'Baris content_clean kosong (hanya emoji/simbol): {n_empty}')
print(f'Total baris difilter: {n_filtered:,} ({n_filtered/n_raw*100:.4f}%)')

# Terapkan filtering
df = df[~mask_filter].reset_index(drop=True)
n_after = len(df)
print(f'Baris tersisa: {n_after:,}')

# Terapkan stopword removal untuk kolom sekunder
df['content_clean_stopword'] = df['content_clean'].apply(remove_stopwords)
print(f'Kolom content_clean_stopword berhasil dibuat!')


Baris content_original NULL: 26
Baris content_clean kosong (hanya emoji/simbol): 7822
Total baris difilter: 7,822 (1.2663%)
Baris tersisa: 609,900


Kolom content_clean_stopword berhasil dibuat!


## 7. Pembuatan Metadata Grup Duplikasi

In [7]:
# Buat duplicate_group berdasarkan content_clean
df['duplicate_group'], _ = pd.factorize(df['content_clean'])
n_groups = df['duplicate_group'].nunique()

group_stats = df.groupby('duplicate_group').agg(
    group_size=('review_id', 'count'),
    n_labels=('label', 'nunique'),
    n_apps_in_group=('app', 'nunique')
)
group_stats['has_label_conflict'] = group_stats['n_labels'] > 1

df = df.merge(
    group_stats[['group_size', 'has_label_conflict', 'n_apps_in_group']],
    left_on='duplicate_group', right_index=True, how='left'
)

n_singleton = int((group_stats['group_size'] == 1).sum())
n_dup_groups = int((group_stats['group_size'] > 1).sum())
n_conflict_groups = int(group_stats['has_label_conflict'].sum())
n_cross_app_groups = int((group_stats['n_apps_in_group'] > 1).sum())

print(f'Grup unik: {n_groups:,}')
print(f'Grup singleton (1 baris): {n_singleton:,}')
print(f'Grup duplikat (>1 baris): {n_dup_groups:,} ({df[df["group_size"]>1].shape[0]:,} baris)')
print(f'Grup dengan konflik label: {n_conflict_groups:,} ({df[df["has_label_conflict"]].shape[0]:,} baris)')
print(f'Grup lintas aplikasi (>1 app): {n_cross_app_groups:,} ({df[df["n_apps_in_group"]>1].shape[0]:,} baris)')


Grup unik: 353,798
Grup singleton (1 baris): 340,859
Grup duplikat (>1 baris): 12,939 (269,041 baris)
Grup dengan konflik label: 3,321 (226,323 baris)
Grup lintas aplikasi (>1 app): 8,688 (256,423 baris)


## 8. Ringkasan Statistik Sebelum dan Sesudah

In [8]:
# Tabel ringkasan
summary = pd.DataFrame({
    'Metrik': [
        'Total baris',
        'Total teks unik (content_clean)',
        'Label Positive', 'Label Neutral', 'Label Negative',
        'App JMO', 'App satusehat', 'App mobileJKN',
        'App pertamina', 'App KAI', 'App BMKG'
    ],
    'Sebelum': [
        n_raw,
        df_raw['content'].nunique(),
        *(df_raw['labelScoreBase'].value_counts().reindex(['Positive','Neutral','Negative']).values),
        *(df_raw['app'].value_counts().reindex(['JMO','satusehat','mobileJKN','pertamina','KAI','BMKG']).values),
    ],
    'Sesudah': [
        n_after,
        n_groups,
        *(df['label'].value_counts().reindex(['Positive','Neutral','Negative']).values),
        *(df['app'].value_counts().reindex(['JMO','satusehat','mobileJKN','pertamina','KAI','BMKG']).values),
    ]
})
summary['Selisih'] = summary['Sebelum'] - summary['Sesudah']
display(summary)

print(f'\nTotal baris difilter: {n_filtered:,} ({n_filtered/n_raw*100:.4f}%)')


,Metrik,Sebelum,Sesudah,Selisih
0,Total baris,617722,609900,7822
1,Total teks unik (content_clean),383614,353798,29816
2,Label Positive,336449,329707,6742
3,Label Neutral,34375,34160,215
4,Label Negative,246898,246033,865
5,App JMO,200000,197216,2784
6,App satusehat,190696,188100,2596
7,App mobileJKN,122103,120497,1606
8,App pertamina,96291,95494,797
9,App KAI,7311,7285,26



Total baris difilter: 7,822 (1.2663%)


## 9. Export Dataset Processed

Menyimpan `data/Rating_labeled_processed.csv` dengan susunan kolom:
`review_id`, `app`, `content_original`, `content_clean`, `content_clean_stopword`, `score`, `label`, `at`, `duplicate_group`, `group_size`, `has_label_conflict`, `n_apps_in_group`.

In [9]:
output_cols = [
    'review_id', 'app', 'content_original', 'content_clean', 'content_clean_stopword',
    'score', 'label', 'at',
    'duplicate_group', 'group_size', 'has_label_conflict', 'n_apps_in_group'
]
df_out = df[output_cols]

# Simpan CSV
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
output_size = os.path.getsize(OUTPUT_CSV) / (1024**2)
print(f'Tersimpan: {OUTPUT_CSV} ({output_size:.1f} MB)')
print(f'Shape: {df_out.shape}')
print(f'Columns: {df_out.columns.tolist()}')
print(f'Null check:')
print(df_out.isnull().sum())


Tersimpan: ../data/Rating_labeled_processed.csv (126.1 MB)
Shape: (609900, 12)
Columns: ['review_id', 'app', 'content_original', 'content_clean', 'content_clean_stopword', 'score', 'label', 'at', 'duplicate_group', 'group_size', 'has_label_conflict', 'n_apps_in_group']
Null check:
review_id                 0
app                       0
content_original          0
content_clean             0
content_clean_stopword    0
score                     0
label                     0
at                        0
duplicate_group           0
group_size                0
has_label_conflict        0
n_apps_in_group           0
dtype: int64
